# Topic 47 | Creating Custom Exceptions in Python

## What is a custom exception?
Python has built-in exceptions (`ValueError`, `FileNotFoundError`, ...), but sometimes none of them describes **our own problem** clearly. A custom exception is an error type that **we create ourselves** for our program, such as `InvalidAgeError` or `InsufficientFundsError`.

## How to create one
A custom exception is a **class that inherits from `Exception`** (this is inheritance from Topic 36 onward):

    class InvalidAgeError(Exception):
        pass

Then we use it like any other exception: `raise` it, and catch it with `except`.

## Why create custom exceptions?
- **Clear names:** `InsufficientFundsError` says more than a generic `ValueError`.
- **Precise handling:** we can catch exactly our error and ignore the others.
- **Extra information:** we can attach data to the error, such as the balance and the amount.
- **Grouping:** related errors can share one parent class, so one `except` catches the whole group.

In [1]:
class InvalidAgeError(Exception):
    pass

def set_age(age):
    if age < 0:
        raise InvalidAgeError("Age cannot be negative")
    return age

try:
    set_age(-5)
except InvalidAgeError as e:
    print("Caught:", e)

Caught: Age cannot be negative


In [4]:
class InsufficientFundsError(Exception):
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"Cannot withdraw Rs.{amount}, balance is only Rs.{balance}")

class Account:
    def __init__(self, balance):
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount
        return self.balance

acc = Account(1000)
print(acc.withdraw(300))                      # 700

try:
    acc.withdraw(5000)
except InsufficientFundsError as e:
    print("Caught:", e)
    print("Short by Rs.", e.amount - e.balance)

# Grouping related errors under one parent
class AppError(Exception): pass
class FileFormatError(AppError): pass
class EmptyFileError(AppError): pass

for exc in [FileFormatError("bad format"), EmptyFileError("empty")]:
    try:
        raise exc
    except AppError as e:                     # one except catches both children
        print(type(e).__name__, "caught by AppError:", e)

700
Caught: Cannot withdraw Rs.5000, balance is only Rs.700
Short by Rs. 4300
FileFormatError caught by AppError: bad format
EmptyFileError caught by AppError: empty


## Rules and good practice
- Inherit from `Exception`, **not** `BaseException`. `BaseException` includes system exits and keyboard interrupts, which you should not catch by accident.
- End the name with **`Error`** (`InvalidAgeError`). It is the Python convention.
- `pass` is enough for a simple one. Add `__init__` only when you want to store extra data.
- Use `super().__init__(message)` so the message works with `print(e)`.
- Make one **base** class for your project (`AppError`) and inherit the others from it. Callers can then catch one specific error or the whole group.
- Create a custom exception only when the built-in ones don't describe the problem well. For a simple bad value, `ValueError` is often enough.

## Key Points
- A custom exception is a **class that inherits from `Exception`**.
- Use `raise` to throw it and `except` to catch it, the same as built-in exceptions.
- It can carry **extra data** (attributes) through its own `__init__`.
- A parent exception class can catch all of its children with a single `except`.
- Clear names make errors easier to understand and debug.